## 💿 DVC를 이용한 데이터 버전 관리

머신러닝 작업을 할 때 데이터 버전을 추적하는 것은 재현성과 일관성을 위해 매우 중요합니다. 데이터는 시간이 지나면서 변하며, 학습이나 평가에 어떤 데이터 버전을 사용했는지 이해하는 것은 문제 해결과 결과 비교를 위한 핵심입니다.

이 노트북에서는 DVC(Data Version Control)를 사용하여 데이터 버전을 효과적으로 관리하는 방법을 살펴봅니다.

### 퀴즈 시간 🤓

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('../.dontlookhere/'))
from quiz5 import *

In [ ]:
quiz_data()

In [ ]:
quiz_versioning()

## 🐠 DVC 설치

먼저 노트북에서 dvc CLI 도구를 사용하기 위해 DVC 의존성을 설치해야 합니다.

In [ ]:
# 의존성 설치
!pip install -q dvc[s3]

## 📽️ 프로젝트 초기화

첫 번째 단계는 DVC 프로젝트를 초기화하는 것입니다. Git 프로젝트 내에서 dvc init을 실행하여 초기화해봅시다:

In [ ]:
# DVC를 초기화합니다. 이것은 캐시, 설정 파일 및 다른 것들을 생성합니다
!cd ..;dvc init

프로젝트에서 초기화되면 DVC는 🧙‍♂️ 설치 디렉토리(.dvc/)에 [내부 디렉토리](https://dvc.org/doc/user-guide/project-structure/internal-files) 및 DVC 작동에 필요한 파일을 생성합니다

In [ ]:
!ls -lhrta ../.dvc

## ⚙️ DVC 원격 저장소 구성

DVC 추적 데이터는 원격 또는 로컬 저장 시스템을 포함하여 다양한 저장 시스템에 업로드할 수 있으며, 이들을 총체적으로 '원격'이라고 부릅니다.

우리의 경우 원격 저장소로 S3를 사용할 것입니다. 두 가지 서로 다른 원격을 구성합니다:

* 실제 데이터를 저장하는 것(s3://data)
* 데이터의 캐시된 버전을 저장하는 것(s3://data-cache)

데이터를 원격 저장소에 푸시하기 전에 dvc remote add 명령을 사용하여 설정해야 합니다. 설명한 대로 먼저 두 개의 서로 다른 원격을 추가하겠습니다. 하나는 데이터 저장용이고 다른 하나는 데이터의 캐시된 버전 저장용입니다(우리의 기본 원격이 될 것입니다):




In [ ]:
# 데이터 버전 관리 저장소를 원격 저장소로 추가합니다
# 이것이 우리의 기본 저장소가 됩니다
!dvc remote add --default s3-version s3://data-cache
!dvc remote modify s3-version endpointurl $AWS_S3_ENDPOINT

In [ ]:
# 데이터 소스를 원격 저장소로 추가합니다
!dvc remote add data-source s3://data
!dvc remote modify data-source endpointurl $AWS_S3_ENDPOINT

.dvc/config 파일은 DVC 구성에 대한 상세한 정보를 포함합니다. 이 파일은 Git으로 추적하도록 설계되었습니다.

검사해보면 각각 다른 데이터 저장소(이 경우 S3 버킷)를 가리키는 두 개의 서로 다른 원격을 정의하고 있음을 알 수 있습니다.

In [ ]:
# 우리의 구성은 이제 이렇게 보입니다
!cat ../.dvc/config

* core 섹션은 일반적인 구성 옵션이 있는 기본 섹션입니다
* 원격 `s3-version`은 데이터의 캐시된 버전을 저장하기 위한 s3 원격을 나타냅니다
* 원격 `data-source`는 데이터를 저장하기 위한 s3 원격을 나타냅니다

## 🐾 데이터 추적

이제 데이터셋을 추적하고 이를 data-cache DVC 원격 저장소로 푸시할 차례입니다!

데이터를 로컬에 저장하지 않을 것입니다. 대신 S3 원격 저장소를 사용하여 데이터를 저장합니다. `dvc import-url` 명령을 사용하면 S3에서 파일을 수동으로 복사하거나 다양한 저장소 유형에 대한 추가 도구를 설치할 필요 없이 외부 데이터 의존성을 만들 수 있습니다.

`--to-remote` 옵션을 사용하면 파일이나 디렉토리를 원격 저장소로 직접 전송하면서 import .dvc 파일을 만들 수 있으므로 효율적이고 간편한 데이터 관리가 가능합니다.

In [ ]:
# 데이터셋을 추적하고 이를 데이터 캐싱 저장소로 푸시합니다
!dvc import-url remote://data-source/song_properties.parquet --to-remote

MinIO의 `data-cache` 버킷 내용을 확인하여 저장된 내용을 볼 수 있습니다.

데이터 추적 과정을 실행한 후 `song_properties.parquet.dvc`라는 새 파일이 생성됩니다. 이 파일에는 방금 추가된 데이터의 특정 버전을 식별하는 DVC 해시가 포함되어 있습니다.

`.dvc` 파일의 구조를 이해하려면 다음 셀을 실행하여 내용을 검사하세요.

In [ ]:
!cat song_properties.parquet.dvc

또한 `.dvc` 파일에 기록된 버전이 MinIO의 `data-cache` 버킷에 저장된 데이터와 일치하는지 확인할 수 있습니다.

MinIO 버킷(`data-cache`)으로 이동하여 해시를 교차 확인하여 일관성을 확인하세요.

## 📁 DVC 파일 관리 및 불필요한 데이터 무시하기

데이터 버전과 코드 버전 간의 명확한 관계를 유지하기 위해 모든 DVC 관련 파일을 Git에 체크인합니다. 이는 프로젝트 전체에서 재현성과 일관성을 보장합니다.

In [ ]:
!git config --global user.email "you@example.com"
!git config --global user.name "Your Name"

!git add song_properties.parquet.dvc .gitignore
!git commit -m "Initial data tracked"

경우에 따라 DVC가 프로젝트 작업 중에 특정 파일을 무시하기를 원할 수도 있습니다. 예를 들어:

- 많은 수의 데이터 파일이 있는 작업 공간에서 작업하면 `dvc status`와 같은 작업의 실행 시간이 길어질 수 있습니다.
- 일부 파일이나 폴더는 프로젝트와 무관할 수 있습니다.

이러한 시나리오를 처리하기 위해 DVC는 Git의 `.gitignore`와 유사하게 작동하는 `.dvcignore` 파일 사용을 지원합니다.